# Cross-Attention (交叉注意力) 实现

交叉注意力用于连接编码器和解码器，是Seq2Seq模型的关键组件。

**关键特点：**
- Query来自解码器（目标序列）
- Key和Value来自编码器（源序列）
- 实现两个不同序列之间的信息交互

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
def softmax(x, axis=-1):
    """Softmax函数"""
    exp_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

## 1. 实现交叉注意力类

In [ ]:
class CrossAttention:
    """
    交叉注意力机制
    
    与Self-Attention的区别：
    - Self-Attention: Q、K、V都来自同一个序列
    - Cross-Attention: Q来自decoder，K、V来自encoder
    """
    
    def __init__(self, embed_dim, use_bias=True):
        self.embed_dim = embed_dim
        self.use_bias = use_bias
        
        # Q用于投影解码器，K、V用于投影编码器
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        
        if use_bias:
            self.b_q = np.zeros(embed_dim)
            self.b_k = np.zeros(embed_dim)
            self.b_v = np.zeros(embed_dim)
    
    def forward(self, decoder_input, encoder_output, mask=None):
        """
        前向传播
        
        Args:
            decoder_input: (tgt_len, embed_dim) - 解码器输入
            encoder_output: (src_len, embed_dim) - 编码器输出
            mask: (tgt_len, src_len) - 可选的mask
        
        Returns:
            output: (tgt_len, embed_dim)
            attention_weights: (tgt_len, src_len)
        """
        # Q来自解码器
        Q = np.dot(decoder_input, self.W_q)
        # K、V来自编码器
        K = np.dot(encoder_output, self.W_k)
        V = np.dot(encoder_output, self.W_v)
        
        if self.use_bias:
            Q += self.b_q
            K += self.b_k
            V += self.b_v
        
        # Scaled Dot-Product Attention
        # (tgt_len, embed_dim) @ (embed_dim, src_len) = (tgt_len, src_len)
        scores = np.dot(Q, K.T) / np.sqrt(self.embed_dim)
        
        # 应用mask
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        
        # Softmax得到注意力权重
        attention_weights = softmax(scores, axis=-1)
        
        # 加权求和
        output = np.dot(attention_weights, V)
        
        return output, attention_weights

## 2. 基础测试：不同长度的序列

In [ ]:
# 参数设置
embed_dim = 64
src_len = 10  # 源序列长度（编码器）
tgt_len = 8   # 目标序列长度（解码器）

# 创建输入
encoder_output = np.random.randn(src_len, embed_dim)
decoder_input = np.random.randn(tgt_len, embed_dim)

# 创建交叉注意力层
cross_attn = CrossAttention(embed_dim)

# 前向传播
output, attn_weights = cross_attn.forward(decoder_input, encoder_output)

print(f"编码器输出形状（源序列）: {encoder_output.shape}")
print(f"解码器输入形状（目标序列）: {decoder_input.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}")
print(f"\n注意：注意力矩阵不是方阵！(tgt_len × src_len)")

## 3. 可视化交叉注意力权重矩阵

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(attn_weights, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': '注意力权重'})
plt.xlabel('源序列位置 (Encoder)', fontsize=12)
plt.ylabel('目标序列位置 (Decoder)', fontsize=12)
plt.title('交叉注意力权重矩阵\n(解码器每个位置关注编码器的权重分布)', fontsize=14)
plt.tight_layout()
plt.show()

print("解读：")
print("- 每一行：解码器某个位置对编码器所有位置的注意力权重")
print("- 每一列：编码器某个位置被解码器所有位置关注的程度")
print("- 每行和为1.0（softmax归一化）")

## 4. 带Padding Mask的交叉注意力

In [ ]:
def create_padding_mask(src_len, tgt_len, valid_src_len):
    """创建padding mask"""
    mask = np.zeros((tgt_len, src_len))
    mask[:, :valid_src_len] = 1
    return mask

# 假设源序列有3个padding
valid_src_len = 7
padding_mask = create_padding_mask(src_len, tgt_len, valid_src_len)

print(f"Padding Mask形状: {padding_mask.shape}")
print(f"有效源序列长度: {valid_src_len}")
print(f"\nMask矩阵（1=有效，0=padding）:")
print(padding_mask.astype(int))

In [ ]:
# 带mask的交叉注意力
output_masked, attn_weights_masked = cross_attn.forward(
    decoder_input, encoder_output, mask=padding_mask
)

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mask
sns.heatmap(padding_mask, annot=True, fmt='.0f', cmap='Greys', ax=axes[0], cbar=False)
axes[0].set_title('Padding Mask', fontsize=12)
axes[0].set_xlabel('源序列位置')
axes[0].set_ylabel('目标序列位置')

# 无mask的注意力
sns.heatmap(attn_weights, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('无Mask的注意力权重', fontsize=12)
axes[1].set_xlabel('源序列位置')
axes[1].set_ylabel('目标序列位置')

# 带mask的注意力
sns.heatmap(attn_weights_masked, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[2])
axes[2].set_title('带Mask的注意力权重', fontsize=12)
axes[2].set_xlabel('源序列位置')
axes[2].set_ylabel('目标序列位置')

# 标记padding区域
axes[2].axvline(x=valid_src_len, color='red', linewidth=2, linestyle='--')
axes[2].text(valid_src_len + 0.5, tgt_len/2, 'Padding\n区域', 
             color='red', fontsize=10, ha='left', va='center')

plt.tight_layout()
plt.show()

print("观察：带mask后，padding位置的注意力权重变为0")

## 5. 机器翻译场景模拟

模拟英译中："I love machine learning" → "我爱机器学习"

In [ ]:
# 定义词汇
src_words = ["I", "love", "machine", "learning", "<PAD>", "<PAD>"]
tgt_words = ["我", "爱", "机器", "学习"]

src_len_mt = len(src_words)
tgt_len_mt = len(tgt_words)
valid_src_len_mt = 4  # 实际有效词数

# 创建词嵌入（随机，实际应该是预训练的）
encoder_output_mt = np.random.randn(src_len_mt, embed_dim)
decoder_input_mt = np.random.randn(tgt_len_mt, embed_dim)

# 创建padding mask
padding_mask_mt = create_padding_mask(src_len_mt, tgt_len_mt, valid_src_len_mt)

# 交叉注意力
output_mt, attn_weights_mt = cross_attn.forward(
    decoder_input_mt, encoder_output_mt, mask=padding_mask_mt
)

print("机器翻译场景：")
print(f"源语言（英文）: {' '.join(src_words)}")
print(f"目标语言（中文）: {' '.join(tgt_words)}")
print(f"\n注意力权重矩阵形状: {attn_weights_mt.shape}")

In [ ]:
# 可视化机器翻译的注意力
plt.figure(figsize=(10, 6))
sns.heatmap(attn_weights_mt, annot=True, fmt='.3f', cmap='YlOrRd', 
            xticklabels=src_words, yticklabels=tgt_words,
            cbar_kws={'label': '注意力权重'})
plt.xlabel('源语言（英文）', fontsize=12)
plt.ylabel('目标语言（中文）', fontsize=12)
plt.title('机器翻译中的交叉注意力\n"I love machine learning" → "我爱机器学习"', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n每个目标词最关注的源词:")
for i, tgt_word in enumerate(tgt_words):
    max_idx = np.argmax(attn_weights_mt[i])
    max_weight = attn_weights_mt[i, max_idx]
    print(f"  '{tgt_word}' 最关注 '{src_words[max_idx]}' (权重: {max_weight:.3f})")

## 6. 分析单个目标词的注意力分布

In [ ]:
# 分析目标词"机器"的注意力
target_pos = 2  # "机器"
attention_dist = attn_weights_mt[target_pos]

plt.figure(figsize=(10, 5))
bars = plt.bar(range(len(src_words)), attention_dist, 
               color=['green' if i < valid_src_len_mt else 'gray' for i in range(len(src_words))])
plt.xticks(range(len(src_words)), src_words, rotation=0)
plt.xlabel('源词（英文）', fontsize=12)
plt.ylabel('注意力权重', fontsize=12)
plt.title(f'目标词 "{tgt_words[target_pos]}" 对源序列的注意力分布', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')
plt.axhline(y=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print(f"目标词 '{tgt_words[target_pos]}' 的注意力权重:")
for i, (word, weight) in enumerate(zip(src_words, attention_dist)):
    status = "(有效)" if i < valid_src_len_mt else "(PAD)"
    print(f"  {word:>10} {status}: {weight:.4f}")

## 7. Cross-Attention vs Self-Attention 对比

In [ ]:
# 创建自注意力进行对比
class SelfAttention:
    def __init__(self, embed_dim):
        self.embed_dim = embed_dim
        self.W_q = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_k = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
        self.W_v = np.random.randn(embed_dim, embed_dim) / np.sqrt(embed_dim)
    
    def forward(self, x):
        Q = np.dot(x, self.W_q)
        K = np.dot(x, self.W_k)
        V = np.dot(x, self.W_v)
        scores = np.dot(Q, K.T) / np.sqrt(self.embed_dim)
        weights = softmax(scores, axis=-1)
        output = np.dot(weights, V)
        return output, weights

# 测试
self_attn = SelfAttention(embed_dim)
x_test = np.random.randn(8, embed_dim)

# Self-Attention
output_self, weights_self = self_attn.forward(x_test)

# Cross-Attention
encoder_out_test = np.random.randn(10, embed_dim)
decoder_in_test = x_test
output_cross, weights_cross = cross_attn.forward(decoder_in_test, encoder_out_test)

# 可视化对比
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Self-Attention
sns.heatmap(weights_self, annot=True, fmt='.2f', cmap='Blues', ax=axes[0])
axes[0].set_title('Self-Attention权重矩阵\n(方阵: 8×8)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Key位置（同一序列）')
axes[0].set_ylabel('Query位置（同一序列）')

# Cross-Attention
sns.heatmap(weights_cross, annot=True, fmt='.2f', cmap='Reds', ax=axes[1])
axes[1].set_title('Cross-Attention权重矩阵\n(非方阵: 8×10)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Key位置（编码器序列）')
axes[1].set_ylabel('Query位置（解码器序列）')

plt.tight_layout()
plt.show()

print("关键区别：")
print(f"{'':20} Self-Attention{' '*10}Cross-Attention")
print("="*70)
print(f"{'Q来源':20} 输入序列{' '*18}解码器")
print(f"{'K来源':20} 输入序列{' '*18}编码器")
print(f"{'V来源':20} 输入序列{' '*18}编码器")
print(f"{'注意力矩阵形状':20} {weights_self.shape}{' '*16}{weights_cross.shape}")
print(f"{'是否为方阵':20} 是{' '*24}否")
print(f"{'用途':20} 序列内部依赖{' '*14}跨序列关联")

## 8. 完整的Encoder-Decoder架构示意

In [ ]:
class EncoderDecoderWithCrossAttention:
    """简化的Encoder-Decoder模型"""
    
    def __init__(self, embed_dim):
        self.embed_dim = embed_dim
        self.cross_attention = CrossAttention(embed_dim)
    
    def forward(self, src_seq, tgt_seq, src_padding_mask=None):
        # 编码器输出（实际会经过多层处理）
        encoder_output = src_seq
        
        # 解码器隐状态（实际会先经过Self-Attention）
        decoder_hidden = tgt_seq
        
        # Cross-Attention: 解码器关注编码器
        output, attention_weights = self.cross_attention.forward(
            decoder_hidden, encoder_output, mask=src_padding_mask
        )
        
        return output, attention_weights

# 测试完整模型
model = EncoderDecoderWithCrossAttention(embed_dim)
src_seq = np.random.randn(10, embed_dim)
tgt_seq = np.random.randn(8, embed_dim)

output_model, attn_model = model.forward(src_seq, tgt_seq)

print("Encoder-Decoder模型测试:")
print(f"源序列形状: {src_seq.shape}")
print(f"目标序列形状: {tgt_seq.shape}")
print(f"输出形状: {output_model.shape}")
print(f"注意力权重形状: {attn_model.shape}")
print("\n✓ Cross-Attention成功连接了编码器和解码器！")

## 9. 总结

### Cross-Attention的核心作用

1. **连接两个序列**：实现编码器到解码器的信息流动
2. **非对称交互**：解码器主动"查询"编码器的信息
3. **长度灵活**：源序列和目标序列长度可以不同

### 典型应用

- 🌐 **机器翻译**：目标语言关注源语言
- 🖼️ **图像描述**：文本关注图像区域
- 🎤 **语音识别**：文本关注音频帧
- ❓ **问答系统**：答案关注问题和文档

### 与Self-Attention的区别

| 特性 | Self-Attention | Cross-Attention |
|------|----------------|------------------|
| Q来源 | 输入序列 | 解码器 |
| K、V来源 | 输入序列 | 编码器 |
| 矩阵形状 | 方阵 (n×n) | 非方阵 (m×n) |
| 用途 | 序列内部依赖 | 跨序列关联 |